# Citiz — collecte des véhicules (GBFS vehicle_status)
Un relevé de tous les véhicules des jeux Citiz de transport.data.gouv.fr, ajouté à la table Delta `citiz_releves`.
**Prérequis :** notebook attaché au lakehouse. **Planification :** toutes les 5 minutes (TTL des flux = 300 s).

In [ ]:
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

TABLE = "citiz_releves"
CATALOGUE = "https://transport.data.gouv.fr/api/datasets"
S = requests.Session()
S.headers["User-Agent"] = "citiz-surveillance/1.0"

In [ ]:
def flux_gbfs():
    jeux = S.get(CATALOGUE, timeout=120).json()
    return {j["slug"]: next(r["url"] for r in j["resources"] if r["format"] == "gbfs")
            for j in jeux
            if "citiz" in j["title"].lower() and any(r["format"] == "gbfs" for r in j["resources"])}


def vehicle_status(item):
    jeu, url = item
    try:
        feeds = {f["name"]: f["url"] for f in S.get(url, timeout=30).json()["data"]["feeds"]}
        vs = S.get(feeds["vehicle_status"], timeout=30).json()["data"]["vehicles"]
    except Exception as e:  # un jeu en panne ne doit pas bloquer les autres
        print(f"{jeu}: ERREUR {e}")
        return []
    return [dict(jeu=jeu, vehicle_id=str(v["vehicle_id"]), lat=v.get("lat"), lon=v.get("lon"),
                 station_id=None if v.get("station_id") is None else str(v["station_id"]),
                 home_station_id=None if v.get("home_station_id") is None else str(v["home_station_id"]),
                 vehicle_type_id=None if v.get("vehicle_type_id") is None else str(v["vehicle_type_id"]),
                 is_reserved=v.get("is_reserved"), is_disabled=v.get("is_disabled"),
                 fuel_pct=v.get("current_fuel_percent"), range_m=v.get("current_range_meters"),
                 last_reported=v.get("last_reported")) for v in vs]

In [ ]:
releve = datetime.now(timezone.utc)
with ThreadPoolExecutor(8) as ex:
    lignes = [r for res in ex.map(vehicle_status, flux_gbfs().items()) for r in res]
if not lignes:
    raise RuntimeError("Aucun véhicule récupéré : flux indisponibles ?")

df = pd.DataFrame(lignes)
df["last_reported"] = pd.to_datetime(df["last_reported"], utc=True, errors="coerce").dt.tz_convert(None)
df["releve_utc"] = releve.replace(tzinfo=None)
print(f"{len(df)} véhicules, {df.jeu.nunique()} jeux")

In [ ]:
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType, BooleanType,
                               TimestampType)

# schéma explicite : une colonne entièrement vide ferait échouer l'inférence
schema = StructType([
    StructField("jeu", StringType()), StructField("vehicle_id", StringType()),
    StructField("lat", DoubleType()), StructField("lon", DoubleType()),
    StructField("station_id", StringType()), StructField("home_station_id", StringType()),
    StructField("vehicle_type_id", StringType()),
    StructField("is_reserved", BooleanType()), StructField("is_disabled", BooleanType()),
    StructField("fuel_pct", DoubleType()), StructField("range_m", DoubleType()),
    StructField("last_reported", TimestampType()), StructField("releve_utc", TimestampType()),
])
sdf = spark.createDataFrame(df.astype(object).where(df.notna(), None), schema=schema)
sdf.write.mode("append").format("delta").saveAsTable(TABLE)
print("Ajouté à", TABLE)

## Maintenance (à exécuter manuellement, pas dans la planification)
Volume : environ 750 000 lignes par jour. La cellule suivante supprime les relevés de plus de 90 jours.

In [ ]:
# spark.sql(f"DELETE FROM {TABLE} WHERE releve_utc < current_timestamp() - INTERVAL 90 DAYS")
# spark.sql(f"OPTIMIZE {TABLE}")